# YOLOv8 Training (RDD2022)

This notebook trains a YOLOv8 model using the preprocessed dataset created in the data prep notebook.

## 1) Install and import

In [ ]:
# Kaggle runtime: install ultralytics
!pip -q install ultralytics

In [ ]:
from pathlib import Path
from ultralytics import YOLO

DATA_YAML = Path("/kaggle/input/datasets/ayush756/dataset-yaml-file/data.yaml")
DATA_YAML

In [ ]:
# Load checkpoint and modify train_args to extend training to 200 epochs
import torch

ckpt = torch.load('/kaggle/input/datasets/ayush756/new-checkpoint/last1.pt', 
                  map_location='cpu', 
                  weights_only=False)

# Check current train_args to confirm
print('original train_args epochs:', ckpt['train_args'].get('epochs'))
print('original epoch:', ckpt['epoch'])

# Fix both — epoch stays 95 (0-indexed = 96th epoch)
# Just update the target epochs in train_args
ckpt['train_args']['epochs'] = 200

# Save modified checkpoint
torch.save(ckpt, '/kaggle/working/last_modified.pt')
print("Saved! Ready to resume to epoch 200")

## 2) Train

In [ ]:
# Choose model size: n, s, m, l, x
MODEL_NAME = "yolov8s.pt"
EPOCHS = 200
IMG_SIZE = 640
BATCH = 16
PROJECT_DIR = "/kaggle/working/yolo_runs"
RUN_NAME = "rdd2022_yolov8s_extend"

ckpt_path = Path("/kaggle/input/datasets/ayush756/weights/last.pt")
if ckpt_path.exists():
    model = YOLO(str(ckpt_path))
    resume_flag = True
else:
    model = YOLO(MODEL_NAME)
    resume_flag = False

results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    project=PROJECT_DIR,
    name=RUN_NAME,
    device="0,1",
    save_period=1,
    cache=False,
    resume=resume_flag,
    patience=50,
    rect=True,
    multi_scale=True,
    cls=0.5,
    box=7.5,
    mosaic=1.0,
    copy_paste=0.3,
    degrees=10.0,
    scale=0.5,
 )
results

## 3) Save best weights path

In [ ]:
best_weights = Path(PROJECT_DIR) / RUN_NAME / "weights" / "best.pt"
best_weights

## 4) Quick sanity inference on train images

In [ ]:
# Find any image regardless of extension
val_folder = Path("/kaggle/input/datasets/aliabdelmenam/rdd-2022/RDD_SPLIT/val/images")

# Try jpg, png, jpeg
sample_img = next(
    img for ext in ['*.jpg', '*.png', '*.jpeg'] 
    for img in val_folder.glob(ext)
)

model = YOLO(str(best_weights))
pred = model.predict(source=str(sample_img), imgsz=IMG_SIZE, conf=0.25, save=True)